# Random Variables, Distributions & Bayes' Rule

This notebook builds a rigorous but intuitive foundation for probability theory as used in probabilistic machine learning and graphical models.

**Topics covered:**
1. Random Variables (discrete & continuous)
2. Joint, Marginal, and Conditional Distributions
3. Bayes' Rule — derivation, intuition, and applications

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch
import seaborn as sns
from scipy import stats
from scipy.stats import norm, binom, poisson, multivariate_normal
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
rng = np.random.default_rng(42)

---
## Part 1 — Random Variables

### 1.1 What is a Random Variable?

A **random variable** $X$ is a function that maps outcomes of a random experiment (a sample space $\Omega$) to real numbers:

$$X : \Omega \to \mathbb{R}$$

The "randomness" lives in $\Omega$; $X$ is just a deterministic measurement of it. For example:

| Experiment | $\Omega$ | Random variable $X$ |
|---|---|---|
| Flip a coin | {H, T} | $X(H)=1,\ X(T)=0$ |
| Roll a die | {1,2,3,4,5,6} | Face value |
| Measure temperature | $\mathbb{R}$ | Temperature in °C |

Random variables come in two flavours:

- **Discrete**: takes a countable set of values — described by a **PMF** (probability mass function)
- **Continuous**: takes values in an interval — described by a **PDF** (probability density function)

### 1.2 Discrete Random Variables — PMF

The **probability mass function** $p_X(x) = P(X = x)$ satisfies:

$$p_X(x) \geq 0 \quad \text{and} \quad \sum_{x} p_X(x) = 1$$

**Key discrete distributions:**

| Distribution | PMF | Parameters |
|---|---|---|
| Bernoulli($p$) | $p^x(1-p)^{1-x}$ | $p \in [0,1]$ |
| Binomial($n,p$) | $\binom{n}{x}p^x(1-p)^{n-x}$ | $n \in \mathbb{N},\ p \in [0,1]$ |
| Poisson($\lambda$) | $\frac{e^{-\lambda}\lambda^x}{x!}$ | $\lambda > 0$ |
| Categorical($\boldsymbol{\pi}$) | $\pi_k$ for class $k$ | $\boldsymbol{\pi}$ simplex |

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Common Discrete Distributions (PMF)', fontweight='bold')

# Binomial
ax = axes[0]
for n, p, color in [(10, 0.3, '#4C72B0'), (10, 0.5, '#DD8452'), (10, 0.7, '#55A868')]:
    x = np.arange(0, n+1)
    ax.plot(x, binom.pmf(x, n, p), 'o-', color=color, label=f'n={n}, p={p}', markersize=5)
ax.set_title('Binomial(n, p)')
ax.set_xlabel('x'); ax.set_ylabel('P(X = x)')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# Poisson
ax = axes[1]
for lam, color in [(1, '#4C72B0'), (4, '#DD8452'), (10, '#55A868')]:
    x = np.arange(0, 20)
    ax.plot(x, poisson.pmf(x, lam), 'o-', color=color, label=f'λ={lam}', markersize=5)
ax.set_title('Poisson(λ)')
ax.set_xlabel('x'); ax.set_ylabel('P(X = x)')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# Categorical
ax = axes[2]
categories = ['A', 'B', 'C', 'D', 'E']
probs = np.array([0.1, 0.35, 0.25, 0.2, 0.1])
bars = ax.bar(categories, probs, color='#4C72B0', alpha=0.8, edgecolor='white')
for bar, p in zip(bars, probs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{p:.2f}', ha='center', va='bottom', fontsize=9)
ax.set_title('Categorical(π)')
ax.set_xlabel('Category'); ax.set_ylabel('Probability')
ax.axhline(1/5, color='red', linestyle='--', linewidth=1, label='Uniform (1/5)')
ax.legend(fontsize=9); ax.grid(alpha=0.3, axis='y')
ax.set_ylim(0, 0.45)

plt.tight_layout()
plt.show()

### 1.3 Continuous Random Variables — PDF & CDF

For a continuous RV, $P(X = x) = 0$ for any single point. Instead we use a **probability density function** $f_X(x)$:

$$P(a \leq X \leq b) = \int_a^b f_X(x)\, dx$$

**Axioms:** $f_X(x) \geq 0$ and $\int_{-\infty}^{\infty} f_X(x)\, dx = 1$.

> **Key insight:** The PDF value itself is NOT a probability — it's a density. You can have $f_X(x) > 1$.

The **cumulative distribution function (CDF)** is defined for both discrete and continuous RVs:
$$F_X(x) = P(X \leq x) = \int_{-\infty}^x f_X(t)\, dt \quad (\text{continuous case})$$

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Gaussian Distribution — PDF, CDF, and Probability as Area', fontweight='bold')

x = np.linspace(-4, 4, 500)

# PDF with shaded probability region
ax = axes[0]
for mu, sigma, color, label in [(0, 1, '#4C72B0', 'μ=0, σ=1'),
                                  (0, 2, '#DD8452', 'μ=0, σ=2'),
                                  (1, 0.5, '#55A868', 'μ=1, σ=0.5')]:
    y = norm.pdf(x, mu, sigma)
    ax.plot(x, y, label=label)

# Shade P(-1 ≤ X ≤ 1) for standard normal
x_fill = np.linspace(-1, 1, 200)
ax.fill_between(x_fill, norm.pdf(x_fill, 0, 1), alpha=0.3, color='#4C72B0',
                label=f'P(-1≤X≤1) = {norm.cdf(1)-norm.cdf(-1):.3f}')
ax.axvline(-1, color='#4C72B0', linestyle=':', alpha=0.7)
ax.axvline(1, color='#4C72B0', linestyle=':', alpha=0.7)
ax.set_title('PDF — probability = area under curve')
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# CDF
ax = axes[1]
for mu, sigma, color, label in [(0, 1, '#4C72B0', 'μ=0, σ=1'),
                                  (0, 2, '#DD8452', 'μ=0, σ=2'),
                                  (1, 0.5, '#55A868', 'μ=1, σ=0.5')]:
    ax.plot(x, norm.cdf(x, mu, sigma), label=label, color=color)
ax.axhline(0.5, color='grey', linestyle='--', linewidth=1, alpha=0.6, label='F(x) = 0.5 (median)')
ax.set_title('CDF — F(x) = P(X ≤ x)')
ax.set_xlabel('x'); ax.set_ylabel('F(x)')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 1.4 Expectation and Variance

The **expected value** (mean) and **variance** summarise the distribution:

$$\mathbb{E}[X] = \sum_x x\, p(x) \quad \text{(discrete)} \qquad \mathbb{E}[X] = \int x\, f(x)\, dx \quad \text{(continuous)}$$

$$\text{Var}(X) = \mathbb{E}[(X - \mathbb{E}[X])^2] = \mathbb{E}[X^2] - (\mathbb{E}[X])^2$$

**Linearity of expectation** (always true, even for dependent RVs):
$$\mathbb{E}[aX + bY] = a\mathbb{E}[X] + b\mathbb{E}[Y]$$

In [ ]:
# Empirically verify E[X] and Var(X) for Gaussian
mu_true, sigma_true = 3.0, 1.5

sample_sizes = [10, 100, 1000, 10000, 100000]
empirical_means = []
empirical_vars = []

for n in sample_sizes:
    samples = rng.normal(mu_true, sigma_true, n)
    empirical_means.append(samples.mean())
    empirical_vars.append(samples.var())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Law of Large Numbers: Empirical Estimates Converging to True Values', fontweight='bold')

for ax, estimates, true_val, label in [
    (axes[0], empirical_means, mu_true, 'E[X] = μ'),
    (axes[1], empirical_vars,  sigma_true**2, 'Var(X) = σ²')
]:
    ax.semilogx(sample_sizes, estimates, 'o-', color='#4C72B0', markersize=7, label='Empirical')
    ax.axhline(true_val, color='red', linestyle='--', linewidth=1.5, label=f'True {label} = {true_val}')
    ax.set_xlabel('Sample size (log scale)')
    ax.set_ylabel(label)
    ax.set_title(f'Estimating {label}')
    ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## Part 2 — Joint, Marginal, and Conditional Distributions

### 2.1 The Joint Distribution

Given two RVs $X$ and $Y$, their **joint distribution** $p(x, y)$ (or $f(x, y)$) captures all probabilistic information about both variables together.

$$\sum_x \sum_y p(x,y) = 1 \qquad \int\!\int f(x,y)\, dx\, dy = 1$$

The joint distribution encodes:
- The behaviour of each variable individually (marginals)
- How the variables relate to each other (dependence structure)

In [ ]:
# --- Build a discrete joint distribution table manually ---
# X = weather {Sunny, Cloudy, Rainy}, Y = mood {Happy, Neutral, Sad}

joint = np.array([
    # Happy  Neutral  Sad
    [0.20,   0.08,   0.02],   # Sunny
    [0.10,   0.10,   0.05],   # Cloudy
    [0.03,   0.07,   0.35],   # Rainy
])

weather_labels = ['Sunny', 'Cloudy', 'Rainy']
mood_labels    = ['Happy', 'Neutral', 'Sad']

print(f'Sum of all joint probabilities: {joint.sum():.2f}')
print()

fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(joint, cmap='Blues', vmin=0)

ax.set_xticks(range(3)); ax.set_yticks(range(3))
ax.set_xticklabels(mood_labels)
ax.set_yticklabels(weather_labels)
ax.set_xlabel('Mood (Y)'); ax.set_ylabel('Weather (X)')
ax.set_title('Joint Distribution p(X, Y)', fontweight='bold')

for i in range(3):
    for j in range(3):
        ax.text(j, i, f'{joint[i,j]:.2f}', ha='center', va='center',
                color='white' if joint[i,j] > 0.15 else 'black', fontsize=12)

plt.colorbar(im, ax=ax, label='Probability')
plt.tight_layout()
plt.show()

### 2.2 Marginal Distributions

The **marginal distribution** of $X$ is obtained by summing (or integrating) over all values of $Y$:

$$p_X(x) = \sum_y p(x, y) \quad \text{("summing out" Y)}$$
$$f_X(x) = \int_{-\infty}^{\infty} f(x, y)\, dy \quad \text{(continuous)}$$

This is called **marginalisation**. The marginal tells you about one variable *regardless* of the other.

In [ ]:
# Compute marginals
p_weather = joint.sum(axis=1)   # marginalise over mood
p_mood    = joint.sum(axis=0)   # marginalise over weather

print('Marginal p(Weather):')
for label, p in zip(weather_labels, p_weather):
    print(f'  p({label}) = {p:.2f}')
print()
print('Marginal p(Mood):')
for label, p in zip(mood_labels, p_mood):
    print(f'  p({label}) = {p:.2f}')

# Visualisation: joint + marginals
fig = plt.figure(figsize=(9, 7))
gs  = gridspec.GridSpec(2, 2, width_ratios=[4, 1.5], height_ratios=[1.5, 4],
                        hspace=0.05, wspace=0.05)

# Joint heatmap
ax_joint = fig.add_subplot(gs[1, 0])
im = ax_joint.imshow(joint, cmap='Blues', vmin=0)
ax_joint.set_xticks(range(3)); ax_joint.set_yticks(range(3))
ax_joint.set_xticklabels(mood_labels)
ax_joint.set_yticklabels(weather_labels)
ax_joint.set_xlabel('Mood (Y)'); ax_joint.set_ylabel('Weather (X)')
for i in range(3):
    for j in range(3):
        ax_joint.text(j, i, f'{joint[i,j]:.2f}', ha='center', va='center',
                     color='white' if joint[i,j] > 0.15 else 'black', fontsize=11)

# Marginal p(Mood) — top bar chart
ax_top = fig.add_subplot(gs[0, 0], sharex=ax_joint)
ax_top.bar(range(3), p_mood, color='#4C72B0', alpha=0.8)
for i, p in enumerate(p_mood):
    ax_top.text(i, p + 0.005, f'{p:.2f}', ha='center', fontsize=9)
ax_top.set_ylabel('p(Mood)')
ax_top.set_title('Joint and Marginal Distributions', fontweight='bold', pad=8)
ax_top.tick_params(labelbottom=False)
ax_top.set_ylim(0, 0.55)
ax_top.grid(alpha=0.3, axis='y')

# Marginal p(Weather) — right bar chart
ax_right = fig.add_subplot(gs[1, 1], sharey=ax_joint)
ax_right.barh(range(3), p_weather, color='#DD8452', alpha=0.8)
for i, p in enumerate(p_weather):
    ax_right.text(p + 0.005, i, f'{p:.2f}', va='center', fontsize=9)
ax_right.set_xlabel('p(Weather)')
ax_right.tick_params(labelleft=False)
ax_right.set_xlim(0, 0.55)
ax_right.grid(alpha=0.3, axis='x')

# Empty corner
fig.add_subplot(gs[0, 1]).axis('off')

plt.show()

### 2.3 Conditional Distributions

The **conditional distribution** of $Y$ given $X = x$ answers: *once I know $X$, how does that change my belief about $Y$?*

$$p(y \mid x) = \frac{p(x, y)}{p(x)} \quad \text{provided } p(x) > 0$$

This is the **product rule** (or chain rule) rearranged:

$$\underbrace{p(x, y)}_{\text{joint}} = \underbrace{p(y \mid x)}_{\text{conditional}} \cdot \underbrace{p(x)}_{\text{marginal}}$$

> **Geometric interpretation:** We slice the joint table at row $x$ and renormalise so the slice sums to 1.

In [ ]:
# Conditional p(Mood | Weather)
# Each row of joint / corresponding marginal p(weather)
conditional_mood_given_weather = joint / p_weather[:, np.newaxis]

print('Conditional p(Mood | Weather):')
header = f'  {"":8}' + ''.join(f'{m:>10}' for m in mood_labels)
print(header)
for weather, row in zip(weather_labels, conditional_mood_given_weather):
    row_str = ''.join(f'{v:10.3f}' for v in row)
    print(f'  {weather:8}{row_str}  (sum={row.sum():.2f})')

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
fig.suptitle('Conditional Distributions p(Mood | Weather = x)', fontweight='bold')

colors = ['#FDB827', '#B0B0B0', '#5B9BD5']
for ax, weather, row in zip(axes, weather_labels, conditional_mood_given_weather):
    bars = ax.bar(mood_labels, row, color=colors, edgecolor='white', alpha=0.9)
    for bar, v in zip(bars, row):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{v:.2f}', ha='center', fontsize=10)
    ax.set_title(f'Weather = {weather}')
    ax.set_xlabel('Mood'); ax.set_ylim(0, 0.9)
    ax.grid(alpha=0.3, axis='y')

axes[0].set_ylabel('p(Mood | Weather)')
plt.tight_layout()
plt.show()

### 2.4 Independence

Two RVs are **independent** ($X \perp Y$) if and only if:

$$p(x, y) = p(x)\, p(y) \quad \text{for all } x, y$$

Equivalent conditions:
- $p(y \mid x) = p(y)$ — knowing $X$ gives no information about $Y$
- $p(x \mid y) = p(x)$ — knowing $Y$ gives no information about $X$
- $\mathbb{E}[XY] = \mathbb{E}[X]\mathbb{E}[Y]$ (and thus $\text{Cov}(X,Y)=0$)

> **Important:** $\text{Cov}(X,Y) = 0$ does **not** imply independence in general. It only implies **linear** independence. Independence implies zero covariance.

In [ ]:
# Illustrate: zero covariance but dependent
theta = rng.uniform(0, 2*np.pi, 2000)
X_circ = np.cos(theta) + rng.normal(0, 0.05, 2000)
Y_circ = np.sin(theta) + rng.normal(0, 0.05, 2000)

# Independent bivariate Gaussian
X_ind = rng.normal(0, 1, 2000)
Y_ind = rng.normal(0, 1, 2000)

# Dependent (positively correlated)
cov_dep = [[1, 0.8], [0.8, 1]]
samples_dep = rng.multivariate_normal([0, 0], cov_dep, 2000)
X_dep, Y_dep = samples_dep[:, 0], samples_dep[:, 1]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Independence vs Dependence', fontweight='bold')

datasets = [
    (X_ind, Y_ind, 'Independent\nCov=0, truly independent'),
    (X_dep, Y_dep, 'Dependent (linear)\nCov>0, r≈0.8'),
    (X_circ, Y_circ, 'Dependent (nonlinear)\nCov≈0 but clearly dependent'),
]

for ax, (X, Y, title) in zip(axes, datasets):
    ax.scatter(X, Y, alpha=0.15, s=5, color='#4C72B0')
    cov = np.cov(X, Y)[0, 1]
    ax.set_title(f'{title}\nEmpirical Cov = {cov:.3f}')
    ax.set_xlabel('X'); ax.set_ylabel('Y')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 2.5 Continuous Joint Distributions — Bivariate Gaussian

The **bivariate Gaussian** is the workhorse of continuous joint distributions:

$$f(x, y) = \frac{1}{2\pi\sigma_x\sigma_y\sqrt{1-\rho^2}} \exp\!\left[-\frac{1}{2(1-\rho^2)}\left(\frac{(x-\mu_x)^2}{\sigma_x^2} - \frac{2\rho(x-\mu_x)(y-\mu_y)}{\sigma_x\sigma_y} + \frac{(y-\mu_y)^2}{\sigma_y^2}\right)\right]$$

where $\rho = \text{Corr}(X,Y) \in [-1, 1]$ is the **correlation coefficient**.

Key property: for Gaussians, the **conditionals and marginals are also Gaussian**.

$$X \mid Y=y \sim \mathcal{N}\!\left(\mu_x + \rho\frac{\sigma_x}{\sigma_y}(y-\mu_y),\; \sigma_x^2(1-\rho^2)\right)$$

In [ ]:
def plot_bivariate_gaussian(rho, ax_3d=None, ax_contour=None):
    mu = [0, 0]
    cov = [[1, rho], [rho, 1]]
    rv  = multivariate_normal(mu, cov)
    
    grid = np.linspace(-3, 3, 100)
    X, Y = np.meshgrid(grid, grid)
    pos  = np.dstack((X, Y))
    Z    = rv.pdf(pos)
    
    if ax_contour is not None:
        cs = ax_contour.contourf(X, Y, Z, levels=15, cmap='Blues')
        # Draw conditional mean line
        y_vals = np.linspace(-3, 3, 100)
        cond_mean_x = rho * y_vals   # E[X|Y=y] with σ_x=σ_y=1, μ=0
        ax_contour.plot(cond_mean_x, y_vals, 'r-', linewidth=2,
                        label=f'E[X|Y=y] = {rho}y')
        ax_contour.set_title(f'ρ = {rho}', fontweight='bold')
        ax_contour.set_xlabel('X'); ax_contour.set_ylabel('Y')
        ax_contour.legend(fontsize=8)
        ax_contour.set_aspect('equal')
    return Z, X, Y

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Bivariate Gaussian — Joint PDF Contours and Conditional Mean', fontweight='bold')

for ax, rho in zip(axes, [-0.8, 0.0, 0.8]):
    plot_bivariate_gaussian(rho, ax_contour=ax)

plt.tight_layout()
plt.show()

In [ ]:
# Show marginals and a conditional slice for ρ=0.7
rho   = 0.7
mu    = np.array([0.0, 0.0])
sigma = np.array([1.0, 1.0])
cov   = np.array([[sigma[0]**2, rho*sigma[0]*sigma[1]],
                  [rho*sigma[0]*sigma[1], sigma[1]**2]])

# Draw samples
samples = rng.multivariate_normal(mu, cov, 3000)
x_s, y_s = samples[:, 0], samples[:, 1]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle(f'Marginals and Conditional Slice (ρ = {rho})', fontweight='bold')

# Marginal of X
ax = axes[0]
ax.hist(x_s, bins=50, density=True, alpha=0.5, color='#4C72B0', label='Samples')
t = np.linspace(-4, 4, 200)
ax.plot(t, norm.pdf(t, mu[0], sigma[0]), 'r-', linewidth=2, label=f'N({mu[0]}, {sigma[0]**2})')
ax.set_title('Marginal p(X)')
ax.set_xlabel('X'); ax.set_ylabel('Density'); ax.legend(); ax.grid(alpha=0.3)

# Marginal of Y
ax = axes[1]
ax.hist(y_s, bins=50, density=True, alpha=0.5, color='#DD8452', label='Samples')
ax.plot(t, norm.pdf(t, mu[1], sigma[1]), 'r-', linewidth=2, label=f'N({mu[1]}, {sigma[1]**2})')
ax.set_title('Marginal p(Y)')
ax.set_xlabel('Y'); ax.set_ylabel('Density'); ax.legend(); ax.grid(alpha=0.3)

# Conditional p(X | Y ≈ 1)
ax = axes[2]
y_target  = 1.0
eps       = 0.15
mask      = np.abs(y_s - y_target) < eps
cond_mean = mu[0] + rho * (sigma[0]/sigma[1]) * (y_target - mu[1])
cond_var  = sigma[0]**2 * (1 - rho**2)

ax.hist(x_s[mask], bins=30, density=True, alpha=0.5, color='#55A868', label=f'Samples with Y≈{y_target}')
ax.plot(t, norm.pdf(t, cond_mean, np.sqrt(cond_var)), 'r-', linewidth=2,
        label=f'N({cond_mean:.2f}, {cond_var:.2f})')
ax.set_title(f'Conditional p(X | Y≈{y_target})')
ax.set_xlabel('X'); ax.set_ylabel('Density'); ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Theoretical conditional mean  E[X|Y={y_target}] = {cond_mean:.3f}')
print(f'Theoretical conditional var Var[X|Y={y_target}] = {cond_var:.3f}')
print(f'Empirical   conditional mean                    = {x_s[mask].mean():.3f}')
print(f'Empirical   conditional var                     = {x_s[mask].var():.3f}')

### 2.6 The Chain Rule of Probability

Any joint distribution can be factored as a chain of conditionals:

$$p(x_1, x_2, \ldots, x_n) = p(x_1)\cdot p(x_2 \mid x_1)\cdot p(x_3 \mid x_1, x_2) \cdots p(x_n \mid x_1, \ldots, x_{n-1})$$

This factorization is the foundation of **autoregressive models** (language models, PixelCNN, etc.). The order of conditioning is arbitrary — but a good ordering (based on causal or natural structure) leads to simpler conditionals.

In [ ]:
# Verify chain rule numerically on the weather-mood joint table
# p(weather, mood) = p(weather) * p(mood | weather)

joint_reconstructed = p_weather[:, np.newaxis] * conditional_mood_given_weather

print('Original joint:')
print(joint)
print()
print('Reconstructed via chain rule p(W)*p(M|W):')
print(np.round(joint_reconstructed, 4))
print()
print(f'Max absolute error: {np.abs(joint - joint_reconstructed).max():.2e}')

---
## Part 3 — Bayes' Rule

### 3.1 Derivation

Bayes' rule follows directly from the definition of conditional probability and the symmetry of the joint distribution:

$$p(x, y) = p(y \mid x)\, p(x) = p(x \mid y)\, p(y)$$

Solving for $p(x \mid y)$:

$$\boxed{p(x \mid y) = \frac{p(y \mid x)\, p(x)}{p(y)}}$$

where the denominator $p(y) = \sum_x p(y \mid x)\, p(x)$ is the **marginal likelihood** (or evidence), computed by marginalising the numerator:

$$p(y) = \sum_x p(y \mid x)\, p(x) \quad\text{(law of total probability)}$$

In the language of **Bayesian inference**, with $H$ = hypothesis and $D$ = data:

$$\underbrace{p(H \mid D)}_{\text{posterior}} = \frac{\overbrace{p(D \mid H)}^{\text{likelihood}} \cdot \overbrace{p(H)}^{\text{prior}}}{\underbrace{p(D)}_{\text{evidence}}}$$

### 3.2 Classic Example — Medical Diagnostic Test

A disease affects **1% of the population**. A test has:
- **Sensitivity** (true positive rate): $P(\text{Test}^+ \mid \text{Disease}) = 0.99$
- **Specificity** (true negative rate): $P(\text{Test}^- \mid \text{No Disease}) = 0.95$

**Question:** If you test positive, what is the probability you actually have the disease?

In [ ]:
def bayes_disease(prevalence, sensitivity, specificity):
    """Compute P(Disease | Test+) via Bayes' rule."""
    p_d   = prevalence
    p_nd  = 1 - prevalence
    p_pos_given_d   = sensitivity          # P(T+ | D)
    p_pos_given_nd  = 1 - specificity      # P(T+ | no D)

    # Law of total probability: P(T+)
    p_pos = p_pos_given_d * p_d + p_pos_given_nd * p_nd

    # Bayes
    posterior = (p_pos_given_d * p_d) / p_pos
    return posterior, p_pos

prevalence  = 0.01
sensitivity = 0.99
specificity = 0.95

posterior, evidence = bayes_disease(prevalence, sensitivity, specificity)

print('=== Medical Test (Bayes\' Rule) ===')
print(f'Prior        P(Disease)         = {prevalence:.4f}  ({prevalence*100:.1f}%)')
print(f'Likelihood   P(T+ | Disease)    = {sensitivity:.4f}  ({sensitivity*100:.1f}%)')
print(f'False Pos.   P(T+ | no Disease) = {1-specificity:.4f}  ({(1-specificity)*100:.1f}%)')
print(f'Evidence     P(T+)              = {evidence:.4f}  ({evidence*100:.2f}%)')
print(f'Posterior    P(Disease | T+)    = {posterior:.4f}  ({posterior*100:.2f}%)')
print()
print('Intuition: Most positives are false positives because the disease is rare.')

In [ ]:
# How does the posterior change with prevalence?
prevalences = np.linspace(0.001, 0.5, 300)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Bayes\' Rule — Medical Test Analysis', fontweight='bold')

ax = axes[0]
for spec, color, label in [(0.95, '#4C72B0', 'Specificity=0.95'),
                            (0.99, '#DD8452', 'Specificity=0.99'),
                            (0.999,'#55A868', 'Specificity=0.999')]:
    posts = [bayes_disease(p, sensitivity, spec)[0] for p in prevalences]
    ax.semilogx(prevalences * 100, posts, label=label)
ax.axvline(1, color='red', linestyle='--', linewidth=1, alpha=0.6, label='1% prevalence')
ax.set_xlabel('Disease Prevalence (%, log scale)')
ax.set_ylabel('P(Disease | Test Positive)')
ax.set_title('Posterior vs Prevalence (sensitivity=0.99)')
ax.legend(); ax.grid(alpha=0.3)

# Visualise the 2x2 confusion table as a tree
ax = axes[1]
ax.axis('off')
p_d   = 0.01
p_tp  = sensitivity * p_d          # P(D, T+)
p_fn  = (1 - sensitivity) * p_d    # P(D, T-)
p_fp  = (1 - specificity) * (1-p_d)# P(no D, T+)
p_tn  = specificity * (1-p_d)      # P(no D, T-)

tree_text = (
    f"Population (N=10,000)\n"
    f"     |\n"
    f"  ┌──┴──────────────────────┐\n"
    f"Disease              No Disease\n"
    f"  100 (1%)             9900 (99%)\n"
    f"  |                    |\n"
    f"┌─┴─┐              ┌───┴──┐\n"
    f"T+  T-            T+      T-\n"
    f"{sensitivity*100:.0f}  {(1-sensitivity)*100:.0f}   \n"
    f"(TP) (FN)         (FP)    (TN)\n"
    f"{p_tp*10000:.0f}   {p_fn*10000:.0f}         {p_fp*10000:.0f}     {p_tn*10000:.0f}\n\n"
    f"P(D|T+) = TP / (TP+FP)\n"
    f"        = {p_tp*10000:.0f} / ({p_tp*10000:.0f} + {p_fp*10000:.0f})\n"
    f"        = {p_tp*10000:.0f} / {(p_tp+p_fp)*10000:.0f}\n"
    f"        ≈ {p_tp/(p_tp+p_fp)*100:.1f}%"
)
ax.text(0.05, 0.95, tree_text, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
ax.set_title('Frequency Tree (N=10,000 people)', fontweight='bold')

plt.tight_layout()
plt.show()

### 3.3 Bayesian Inference — Sequential Updating

One of the most powerful aspects of Bayes' rule: **yesterday's posterior is today's prior**.

As we observe data $D_1, D_2, \ldots$, we update our beliefs sequentially:

$$p(\theta \mid D_1) \propto p(D_1 \mid \theta)\, p(\theta)$$
$$p(\theta \mid D_1, D_2) \propto p(D_2 \mid \theta)\, p(\theta \mid D_1)$$

**Example:** Estimating the bias $\theta$ of a coin. Prior: $\theta \sim \text{Beta}(\alpha, \beta)$. Each flip is Bernoulli($\theta$). The Beta–Bernoulli conjugate pair gives a closed-form posterior:

$$\theta \mid \text{data} \sim \text{Beta}(\alpha + \#\text{heads},\ \beta + \#\text{tails})$$

In [ ]:
true_theta = 0.7   # true bias (unknown to the statistician)
flips = rng.binomial(1, true_theta, 200)   # simulate coin flips

# Prior: Beta(2, 2) — slight belief towards fair coin
alpha0, beta0 = 2, 2

theta_vals = np.linspace(0, 1, 500)
checkpoints = [0, 1, 3, 10, 30, 100, 200]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
fig.suptitle(f'Sequential Bayesian Updating — Coin Bias θ (true θ = {true_theta})', fontweight='bold')
axes = axes.flatten()

for i, n_obs in enumerate(checkpoints):
    ax = axes[i]
    heads = flips[:n_obs].sum()
    tails = n_obs - heads

    alpha_n = alpha0 + heads
    beta_n  = beta0  + tails

    prior_pdf     = stats.beta.pdf(theta_vals, alpha0, beta0)
    posterior_pdf = stats.beta.pdf(theta_vals, alpha_n, beta_n)
    posterior_mean = alpha_n / (alpha_n + beta_n)

    ax.plot(theta_vals, prior_pdf, '--', color='grey', linewidth=1.2, label='Prior', alpha=0.7)
    ax.plot(theta_vals, posterior_pdf, color='#4C72B0', linewidth=2,
            label=f'Posterior\nBeta({alpha_n},{beta_n})')
    ax.axvline(true_theta, color='red', linestyle=':', linewidth=1.5, label=f'True θ={true_theta}')
    ax.axvline(posterior_mean, color='#4C72B0', linestyle='--', linewidth=1,
               label=f'Post.mean={posterior_mean:.2f}')
    ax.fill_between(theta_vals, posterior_pdf, alpha=0.15, color='#4C72B0')
    ax.set_title(f'After {n_obs} flips\n(H={heads}, T={tails})')
    ax.set_xlabel('θ'); ax.set_ylim(bottom=0)
    ax.legend(fontsize=7); ax.grid(alpha=0.2)

axes[-1].axis('off')
plt.tight_layout()
plt.show()

### 3.4 Bayes' Rule for Continuous Parameters — Gaussian Update

Suppose we're estimating the mean $\mu$ of a Gaussian with known variance $\sigma^2 = 1$.

- **Prior:** $\mu \sim \mathcal{N}(\mu_0, \tau_0^2)$
- **Likelihood:** $x_i \mid \mu \sim \mathcal{N}(\mu, \sigma^2)$

After observing $n$ data points with sample mean $\bar{x}$, the posterior is:

$$\mu \mid \mathbf{x} \sim \mathcal{N}(\mu_n, \tau_n^2)$$

$$\frac{1}{\tau_n^2} = \frac{1}{\tau_0^2} + \frac{n}{\sigma^2} \qquad \mu_n = \tau_n^2\left(\frac{\mu_0}{\tau_0^2} + \frac{n\bar{x}}{\sigma^2}\right)$$

The posterior mean is a **precision-weighted average** of the prior mean and the data mean — more data means more weight on the data.

In [ ]:
true_mu = 3.0
sigma2  = 1.0     # known likelihood variance

# Prior: believe mu ≈ 0 with some uncertainty
mu0   = 0.0
tau0  = 2.0

mu_vals = np.linspace(-3, 7, 400)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Gaussian–Gaussian Conjugate Update: Estimating the Mean μ', fontweight='bold')

for ax, n_obs in zip(axes, [0, 5, 50]):
    data  = rng.normal(true_mu, np.sqrt(sigma2), n_obs)
    xbar  = data.mean() if n_obs > 0 else 0

    # Posterior parameters
    prec_prior = 1 / tau0**2
    prec_data  = n_obs / sigma2
    prec_post  = prec_prior + prec_data
    tau_n      = np.sqrt(1 / prec_post)
    mu_n       = (prec_prior * mu0 + prec_data * xbar) / prec_post if n_obs > 0 else mu0

    prior_pdf = norm.pdf(mu_vals, mu0, tau0)
    post_pdf  = norm.pdf(mu_vals, mu_n, tau_n)

    ax.plot(mu_vals, prior_pdf, '--', color='grey', label=f'Prior N({mu0},{tau0}²)', linewidth=1.5)
    ax.plot(mu_vals, post_pdf, color='#4C72B0', linewidth=2.5,
            label=f'Posterior N({mu_n:.2f},{tau_n**2:.2f})')
    ax.fill_between(mu_vals, post_pdf, alpha=0.15, color='#4C72B0')
    ax.axvline(true_mu, color='red', linestyle=':', linewidth=2, label=f'True μ={true_mu}')
    if n_obs > 0:
        ax.axvline(xbar, color='green', linestyle='--', linewidth=1.5,
                   label=f'MLE x̄={xbar:.2f}')
    ax.set_title(f'n = {n_obs} observations')
    ax.set_xlabel('μ'); ax.set_ylabel('Density')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 3.5 Bayes' Rule — Classifier (Naive Bayes)

Bayes' rule underlies classification. Given features $\mathbf{x}$ and class label $C$:

$$p(C \mid \mathbf{x}) = \frac{p(\mathbf{x} \mid C)\, p(C)}{p(\mathbf{x})}$$

**Naive Bayes** assumes feature independence given the class:
$$p(\mathbf{x} \mid C) = \prod_{j} p(x_j \mid C)$$

For prediction, we compare posteriors across classes and pick the largest — the normalising constant $p(\mathbf{x})$ cancels:

$$\hat{C} = \arg\max_C\ p(C)\prod_j p(x_j \mid C)$$

In [ ]:
# Gaussian Naive Bayes on 2D data
from sklearn.datasets import make_classification
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split

X_raw, y_raw = make_classification(n_samples=800, n_features=2, n_redundant=0,
                                    n_informative=2, n_clusters_per_class=1,
                                    class_sep=1.5, random_state=0)

X_tr, X_te, y_tr, y_te = train_test_split(X_raw, y_raw, test_size=0.3, random_state=0)
gnb = GaussianNB()
gnb.fit(X_tr, y_tr)
accuracy = gnb.score(X_te, y_te)

# Decision boundary via posterior
xx, yy = np.meshgrid(np.linspace(X_raw[:,0].min()-1, X_raw[:,0].max()+1, 300),
                     np.linspace(X_raw[:,1].min()-1, X_raw[:,1].max()+1, 300))
Z = gnb.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Gaussian Naive Bayes — Bayes\' Rule as a Classifier', fontweight='bold')

ax = axes[0]
ax.contourf(xx, yy, Z, levels=50, cmap='RdBu_r', alpha=0.6)
ax.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=1.5)
scatter = ax.scatter(X_te[:, 0], X_te[:, 1], c=y_te, cmap='RdBu_r', edgecolors='k',
                     linewidths=0.3, s=40, alpha=0.9)
ax.set_title(f'Decision Boundary (Test Accuracy = {accuracy:.3f})')
ax.set_xlabel('Feature 1'); ax.set_ylabel('Feature 2')
plt.colorbar(scatter, ax=ax, label='True Class')
ax.grid(alpha=0.2)

# Posterior probability for a slice
ax = axes[1]
x1_fixed = 0.0
x2_range = np.linspace(X_raw[:, 1].min() - 1, X_raw[:, 1].max() + 1, 300)
proba_slice = gnb.predict_proba(np.c_[np.full_like(x2_range, x1_fixed), x2_range])

ax.plot(x2_range, proba_slice[:, 0], '#DD8452', linewidth=2, label='P(Class=0 | x)')
ax.plot(x2_range, proba_slice[:, 1], '#4C72B0', linewidth=2, label='P(Class=1 | x)')
ax.axhline(0.5, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax.fill_between(x2_range, proba_slice[:, 1], 0.5,
                where=proba_slice[:, 1] > 0.5, alpha=0.1, color='#4C72B0')
ax.fill_between(x2_range, proba_slice[:, 0], 0.5,
                where=proba_slice[:, 0] > 0.5, alpha=0.1, color='#DD8452')
ax.set_title(f'Posterior P(Class | Feature1={x1_fixed}, Feature2=x₂)')
ax.set_xlabel('Feature 2 (x₂)'); ax.set_ylabel('Posterior Probability')
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 3.6 The Base Rate Fallacy

Humans systematically ignore **prior probabilities** — this is the **base rate fallacy**. Bayes' rule shows exactly how to integrate priors:

The posterior depends on both how good the evidence is (likelihood ratio) AND how likely the hypothesis was before (prior).

$$\underbrace{\frac{p(H \mid D)}{p(\neg H \mid D)}}_{\text{posterior odds}} = \underbrace{\frac{p(D \mid H)}{p(D \mid \neg H)}}_{\text{likelihood ratio (Bayes factor)}} \times \underbrace{\frac{p(H)}{p(\neg H)}}_{\text{prior odds}}$$

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

bayes_factor = 0.99 / 0.05   # sensitivity / false positive rate
prior_odds_range = np.logspace(-4, 1, 500)  # prior odds P(H)/(1-P(H))
posterior_odds   = bayes_factor * prior_odds_range
posterior_proba  = posterior_odds / (1 + posterior_odds)
prior_proba      = prior_odds_range / (1 + prior_odds_range)

ax.semilogx(prior_proba * 100, posterior_proba * 100, '#4C72B0', linewidth=2.5)

# Annotate key points
for prev, marker_color in [(0.001, '#55A868'), (0.01, '#DD8452'), (0.1, '#C44E52'), (0.5, '#8172B3')]:
    odds  = prev / (1 - prev)
    post  = bayes_factor * odds / (1 + bayes_factor * odds) * 100
    ax.scatter([prev * 100], [post], s=100, color=marker_color, zorder=5)
    ax.annotate(f'Prior={prev*100:.1f}%\n→ Post={post:.1f}%',
                xy=(prev*100, post), xytext=(20, -20), textcoords='offset points',
                fontsize=8, color=marker_color,
                arrowprops=dict(arrowstyle='->', color=marker_color, lw=1.2))

ax.axline((0, 0), slope=1, color='grey', linestyle='--', linewidth=1, label='No update (Bayes factor=1)')
ax.set_xlabel('Prior Probability P(Disease) [%]')
ax.set_ylabel('Posterior Probability P(Disease|T+) [%]')
ax.set_title(f'Impact of Prior on Posterior\n(Bayes Factor = {bayes_factor:.1f}, sensitivity=0.99, FPR=0.05)',
             fontweight='bold')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## Summary Table

| Concept | Formula | What it answers |
|---|---|---|
| PMF / PDF | $p(x)$, $f(x)$ | How probable is value $x$? |
| CDF | $F(x) = P(X \leq x)$ | Cumulative probability up to $x$ |
| Joint | $p(x, y)$ | Probability of both $X=x$ **and** $Y=y$ |
| Marginal | $p(x) = \sum_y p(x,y)$ | Probability of $X=x$ ignoring $Y$ |
| Conditional | $p(y\mid x) = p(x,y)/p(x)$ | Probability of $Y=y$ **given** $X=x$ |
| Chain rule | $p(x,y) = p(y\mid x)p(x)$ | Decompose joint into conditional × marginal |
| Independence | $p(x,y) = p(x)p(y)$ | $X$ and $Y$ carry no info about each other |
| Bayes' rule | $p(x\mid y) = p(y\mid x)p(x)/p(y)$ | Reverse the conditioning |

---
## Key Takeaways

1. **Random variables** are functions on a sample space; randomness enters only through the underlying experiment.
2. **Joint distributions** encode all pairwise (and higher-order) relationships. Every other distribution (marginal, conditional) is derived from it.
3. **Marginalisation** (summing/integrating out a variable) reduces dimensionality — it's the engine behind latent variable models.
4. **Conditioning** updates a distribution given observed evidence — it is the foundation of inference.
5. **Bayes' rule** lets us invert conditional distributions: go from $p(\text{data}\mid\theta)$ (model) to $p(\theta\mid\text{data})$ (belief after observing data).
6. **Base rates matter** — the prior $p(H)$ can dominate even a strong likelihood when the prior is very small.